In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

HORIZONS = [12, 24, 48, 72]

In [2]:
# ─────────────────────────────────────────────
# 2. SURVIVAL HELPERS
# ─────────────────────────────────────────────
def y_horizon(df, H):
    if H == 72:
        return df["event"].astype(int)
    return ((df["event"] == 1) & (df["time_to_hit_hours"] <= H)).astype(int)

def known_mask(df, H):
    if H == 72:
        return np.ones(len(df), dtype=bool)
    return ~((df["event"] == 0) & (df["time_to_hit_hours"] < H))

def weighted_brier(df, probs_df, horizons=(24, 48, 72), weights={24:0.3, 48:0.4, 72:0.3}):
    from sklearn.metrics import brier_score_loss
    briers = {}
    for H in horizons:
        col = f"prob_{H}h"
        m = known_mask(df, H)
        y_true = y_horizon(df, H)[m]
        y_pred = probs_df.loc[m, col]
        briers[H] = brier_score_loss(y_true, y_pred)
    wb = sum(weights[H] * briers[H] for H in horizons)
    return wb, briers

def enforce_monotonic(df_probs):
    out = df_probs.copy()
    out["prob_24h"] = np.maximum(out["prob_24h"], out["prob_12h"])
    out["prob_48h"] = np.maximum(out["prob_48h"], out["prob_24h"])
    out["prob_72h"] = np.maximum(out["prob_72h"], out["prob_48h"])
    return out.clip(0, 1)

In [3]:
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────
def feature_engineering(df, train_ref=None):
    """
    Focused on the 4 features that actually matter.
    train_ref: pass training df to compute rank features on test set.
    """
    df = df.copy()
    eps = 1e-6

    # ── Core raw signals ──────────────────────────────────────────
    dist    = df["dist_min_ci_0_5h"].clip(lower=1)
    dist_km = dist / 1000.0
    area    = df["area_first_ha"].clip(lower=eps)
    n_perim = df["num_perimeters_0_5h"]
    dt      = df["dt_first_last_0_5h"].clip(lower=eps)

    # Derived speed (the raw closing/radial were zero-importance —
    # but speed inferred from area/perimeter/time might be better)
    closing = df["closing_speed_m_per_h"]
    radial  = df["radial_growth_rate_m_per_h"]
    eff_spd = (closing + radial).clip(lower=0)

    # ── Deep transforms of dist_min (most important feature) ──────
    df["dist_km"]          = dist_km
    df["log_dist"]         = np.log1p(dist_km)
    df["log_dist_sq"]      = df["log_dist"] ** 2
    df["sqrt_dist"]        = np.sqrt(dist_km)
    df["inv_dist"]         = 1.0 / (dist_km + 0.1)
    df["inv_dist_sq"]      = 1.0 / (dist_km ** 2 + 0.1)
    df["zone_critical_1k"] = (dist < 1000).astype(int)
    df["zone_critical_5k"] = (dist < 5000).astype(int)
    df["zone_critical_10k"]= (dist < 10000).astype(int)

    # ── Deep transforms of area_first_ha ─────────────────────────
    df["log_area"]         = np.log1p(area)
    df["sqrt_area"]        = np.sqrt(area)
    df["log_area_sq"]      = df["log_area"] ** 2
    fire_radius            = np.sqrt(area * 10000 / np.pi)
    df["fire_radius_km"]   = fire_radius / 1000.0
    df["radius_dist_ratio"]= fire_radius / np.maximum(dist, eps)
    df["log_radius_dist"]  = np.log1p(df["radius_dist_ratio"])

    # ── Deep transforms of num_perimeters ────────────────────────
    df["log_n_perim"]      = np.log1p(n_perim)
    df["n_perim_sq"]       = n_perim ** 2
    # More perimeters = longer observation = more data on this fire
    df["area_per_perim"]   = area / np.maximum(n_perim, 1)
    df["log_area_per_perim"]= np.log1p(df["area_per_perim"])

    # ── Deep transforms of dt_first_last ─────────────────────────
    df["log_dt"]           = np.log1p(dt)
    # Growth rate: area gained per unit time observed
    df["area_growth_rate"] = area / dt
    df["log_growth_rate"]  = np.log1p(df["area_growth_rate"])
    # Speed of perimeter change
    df["perim_rate"]       = n_perim / dt
    df["log_perim_rate"]   = np.log1p(df["perim_rate"])

    # ── Cross-feature interactions (top 4 only) ───────────────────
    df["dist_x_log_area"]  = dist_km * df["log_area"]
    df["inv_dist_x_area"]  = df["inv_dist"] * df["log_area"]
    df["growth_x_inv_dist"]= df["log_growth_rate"] * df["inv_dist"]
    df["area_x_perim"]     = df["log_area"] * df["log_n_perim"]
    df["dist_x_perim"]     = df["log_dist"] * df["log_n_perim"]
    df["dist_x_dt"]        = dist_km * df["log_dt"]

    # ── Keep dist_margin_12h (only engineered feature with signal) ─
    df["dist_margin_12h"]  = (eff_spd * 12 - dist) / 1000.0
    df["dist_margin_24h"]  = (eff_spd * 24 - dist) / 1000.0

    # ── Keep other raw columns that had any importance ────────────
    # (zone_critical, radius_to_distance already covered above)
    # Keep month/dayofweek if they were in original
    # (event_start_month, event_start_dayofweek — already in raw cols)

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df


train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)

# ── Rank features (computed post-hoc using training distribution) ─
# These are robust on small datasets — no distribution assumptions
rank_cols = ["dist_km", "log_area", "log_growth_rate", 
             "inv_dist", "radius_dist_ratio"]

for col in rank_cols:
    train_fe[f"{col}_rank"] = train_fe[col].rank(pct=True)
    # Test ranks relative to training distribution
    test_fe[f"{col}_rank"]  = test_fe[col].apply(
        lambda x, c=col: (train_fe[c] < x).mean()
    )

FEATURE_COLS = [
    # The 4 dominant raw features
    "dist_min_ci_0_5h",
    "area_first_ha", 
    "num_perimeters_0_5h",
    "dt_first_last_0_5h",
    
    # Raw cols with consistent small importance
    "low_temporal_resolution_0_5h",
    "dist_slope_ci_0_5h",
    "event_start_month",
    "event_start_dayofweek",
    
    # The one engineered feature that consistently shows up
    "dist_margin_12h",
    
    # Transforms that appeared at least once
    "zone_critical_5k",
    "radius_dist_ratio",
    "area_per_perim",
    "log_area_per_perim",
    "area_growth_rate",
    "area_x_perim",
    "dist_margin_24h",
]

print(f"Lean feature set: {len(FEATURE_COLS)} features")

Lean feature set: 16 features


In [4]:
from lightgbm import LGBMClassifier
# Quick importance re-check after new features
model_imp = LGBMClassifier(objective="binary", n_estimators=400,
                            learning_rate=0.04, max_depth=3,
                            num_leaves=11, n_jobs=-1, verbose=-1)
m72 = known_mask(train_fe, 72)
model_imp.fit(train_fe.loc[m72, FEATURE_COLS], y_horizon(train_fe, 72).loc[m72])

imp = pd.Series(model_imp.feature_importances_, index=FEATURE_COLS)
top20 = imp.sort_values(ascending=False).head(20)
print(top20.to_string())

# Count how many non-raw features appear in top 10
raw_feats = ["dist_min_ci_0_5h", "area_first_ha", 
             "num_perimeters_0_5h", "dt_first_last_0_5h"]
top10 = imp.sort_values(ascending=False).head(10)
engineered_in_top10 = [f for f in top10.index if f not in raw_feats]
print(f"\nEngineered features in top 10: {engineered_in_top10}")

dist_min_ci_0_5h                550
area_first_ha                    66
dist_margin_12h                  21
zone_critical_5k                  6
dt_first_last_0_5h                5
radius_dist_ratio                 2
dist_slope_ci_0_5h                1
event_start_month                 1
area_per_perim                    1
area_growth_rate                  1
area_x_perim                      1
num_perimeters_0_5h               0
low_temporal_resolution_0_5h      0
event_start_dayofweek             0
log_area_per_perim                0
dist_margin_24h                   0

Engineered features in top 10: ['dist_margin_12h', 'zone_critical_5k', 'radius_dist_ratio', 'dist_slope_ci_0_5h', 'event_start_month', 'area_per_perim', 'area_growth_rate']


In [5]:
# ─────────────────────────
# 4. LGBM SEED BAGGING 
# ─────────────────────────
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold

print("=== Quick param grid search ===")
param_grid = [
    dict(max_depth=2, num_leaves=7,  n_estimators=600, learning_rate=0.03),
    dict(max_depth=3, num_leaves=11, n_estimators=400, learning_rate=0.04),  # original
    dict(max_depth=3, num_leaves=15, n_estimators=500, learning_rate=0.03),
    dict(max_depth=4, num_leaves=15, n_estimators=400, learning_rate=0.04),
    dict(max_depth=2, num_leaves=15, n_estimators=600, learning_rate=0.03),
    dict(max_depth=4, num_leaves=31, n_estimators=300, learning_rate=0.05),
]
BASE_FIXED = dict(min_child_samples=20, reg_lambda=2.0,
                  subsample=0.9, colsample_bytree=0.8,
                  objective="binary", n_jobs=-1, verbose=-1)

best_wb     = 999
best_params = param_grid[1]

_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for p in param_grid:
    _oof_raw = pd.DataFrame(0.0, index=train_fe.index,
                            columns=[f"prob_{H}h" for H in HORIZONS])
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        XH  = train_fe.loc[m, FEATURE_COLS]
        yH  = y_horizon(train_fe, H).loc[m]
        for fold, (tr_i, va_i) in enumerate(_cv.split(XH, yH)):
            mdl = LGBMClassifier(**BASE_FIXED, **p)
            mdl.fit(XH.iloc[tr_i], yH.iloc[tr_i],
                    eval_set=[(XH.iloc[va_i], yH.iloc[va_i])],
                    eval_metric="binary_logloss",
                    callbacks=[early_stopping(50, verbose=False),
                               log_evaluation(0)])
            _oof_raw.loc[XH.index[va_i], col] = \
                mdl.predict_proba(XH.iloc[va_i])[:, 1]

    # ← apply isotonic so grid search matches the seed loop pipeline exactly
    _oof_cal = _oof_raw.copy()
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(_oof_raw.loc[m, col], y_horizon(train_fe, H).loc[m])
        _oof_cal[col] = iso.transform(_oof_raw[col])
    _oof_cal = enforce_monotonic(_oof_cal)

    wb, _ = weighted_brier(train_fe, _oof_cal)
    marker = " ← best" if wb < best_wb else ""
    print(f"  depth={p['max_depth']} leaves={p['num_leaves']:2d} "
          f"lr={p['learning_rate']} est={p['n_estimators']} → WB={wb:.6f}{marker}")
    if wb < best_wb:
        best_wb     = wb
        best_params = p

print(f"\nBest params: {best_params}  (WB={best_wb:.6f})\n")

# ── Seed loop config ──────────────────────────────────────────────
SEEDS = list(range(42, 72))  # 30 seeds
BEST_PARAMS = {
    "n_estimators":      best_params["n_estimators"],
    "learning_rate":     best_params["learning_rate"],
    "max_depth":         best_params["max_depth"],
    "num_leaves":        best_params["num_leaves"],
    "min_child_samples": 20,
    "reg_lambda":        2.0,
}

test_sum  = pd.DataFrame(0.0, index=test_fe.index,  columns=[f"prob_{H}h" for H in HORIZONS])
oof_sum   = pd.DataFrame(0.0, index=train_fe.index, columns=[f"prob_{H}h" for H in HORIZONS])
seed_scores = []

for seed in SEEDS:
    print(f"\n===== LGBM SEED {seed} =====")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof_raw  = pd.DataFrame(0.0, index=train_fe.index, columns=[f"prob_{H}h" for H in HORIZONS])
    test_raw = pd.DataFrame(0.0, index=test_fe.index,  columns=[f"prob_{H}h" for H in HORIZONS])

    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        XH  = train_fe.loc[m, FEATURE_COLS]
        yH  = y_horizon(train_fe, H).loc[m]

        for fold, (tr_idx, va_idx) in enumerate(cv.split(XH, yH), 1):
            model = LGBMClassifier(
                objective="binary", subsample=0.9, colsample_bytree=0.8,
                random_state=seed + 1000*H + fold, n_jobs=-1, verbose=-1,
                **BEST_PARAMS
            )
            model.fit(
                XH.iloc[tr_idx], yH.iloc[tr_idx],
                eval_set=[(XH.iloc[va_idx], yH.iloc[va_idx])],
                eval_metric="binary_logloss",
                callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
            )
            va_index = XH.index[va_idx]
            oof_raw.loc[va_index, col] = model.predict_proba(XH.iloc[va_idx])[:, 1]
            test_raw[col] += model.predict_proba(test_fe[FEATURE_COLS])[:, 1] / cv.get_n_splits()

    # Isotonic on ALL horizons including 72h (Bug 1 fix)
    oof_cal  = oof_raw.copy()
    test_cal = test_raw.copy()
    for H in HORIZONS:
        col = f"prob_{H}h"
        m   = known_mask(train_fe, H)
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_raw.loc[m, col], y_horizon(train_fe, H).loc[m])
        oof_cal[col]  = iso.transform(oof_raw[col])
        test_cal[col] = iso.transform(test_raw[col])

    oof_cal  = enforce_monotonic(oof_cal)
    test_cal = enforce_monotonic(test_cal)

    wb, briers = weighted_brier(train_fe, oof_cal)
    seed_scores.append(wb)
    print(f"  OOF WB: {wb:.6f} | {briers}")

    test_sum += test_cal
    oof_sum  += oof_cal

lgbm_test_bagged = enforce_monotonic(test_sum / len(SEEDS))
lgbm_oof_bagged  = enforce_monotonic(oof_sum  / len(SEEDS))
lgbm_wb, _       = weighted_brier(train_fe, lgbm_oof_bagged)
print(f"\nLGBM final OOF WB: {lgbm_wb:.6f}  (target: beat 0.009075)")

=== Quick param grid search ===
  depth=2 leaves= 7 lr=0.03 est=600 → WB=0.011484 ← best
  depth=3 leaves=11 lr=0.04 est=400 → WB=0.011739
  depth=3 leaves=15 lr=0.03 est=500 → WB=0.011764
  depth=4 leaves=15 lr=0.04 est=400 → WB=0.011759
  depth=2 leaves=15 lr=0.03 est=600 → WB=0.011484
  depth=4 leaves=31 lr=0.05 est=300 → WB=0.011217 ← best

Best params: {'max_depth': 4, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.05}  (WB=0.011217)


===== LGBM SEED 42 =====
  OOF WB: 0.011578 | {24: 0.020904820511638028, 48: 0.013265502870177128, 72: 0.0}

===== LGBM SEED 43 =====
  OOF WB: 0.012608 | {24: 0.02303157969069981, 48: 0.012338448372498067, 72: 0.0025452488687782806}

===== LGBM SEED 44 =====
  OOF WB: 0.016464 | {24: 0.027013029324155696, 48: 0.018469736935909316, 72: 0.0032397119066106188}

===== LGBM SEED 45 =====
  OOF WB: 0.014403 | {24: 0.025087759462759464, 48: 0.014594497318663214, 72: 0.003464366515837104}

===== LGBM SEED 46 =====
  OOF WB: 0.013350 | {24: 0.022

In [6]:
# ── DIAGNOSTIC: run this immediately after your seed bagging loop ──

print("\n=== PREDICTION VARIANCE CHECK ===")
for H in HORIZONS:
    col = f"prob_{H}h"
    preds = lgbm_oof_bagged[col]
    print(f"\nH={H}h:")
    print(f"  mean={preds.mean():.4f}  std={preds.std():.4f}")
    print(f"  min={preds.min():.4f}  max={preds.max():.4f}")
    print(f"  % near zero (<0.01): {(preds < 0.01).mean()*100:.1f}%")
    print(f"  % near one  (>0.99): {(preds > 0.99).mean()*100:.1f}%")

print("\n=== EVENT RATE BY HORIZON ===")
for H in HORIZONS:
    m = known_mask(train_fe, H)
    y = y_horizon(train_fe, H)
    rate = y[m].mean()
    n_hits = y[m].sum()
    n_total = m.sum()
    print(f"H={H}h: {n_hits} hits / {n_total} known = {rate:.4f} event rate")


=== PREDICTION VARIANCE CHECK ===

H=12h:
  mean=0.2217  std=0.3565
  min=0.0000  max=1.0000
  % near zero (<0.01): 68.3%
  % near one  (>0.99): 3.6%

H=24h:
  mean=0.2911  std=0.4339
  min=0.0000  max=1.0000
  % near zero (<0.01): 68.3%
  % near one  (>0.99): 11.3%

H=48h:
  mean=0.3058  std=0.4531
  min=0.0000  max=1.0000
  % near zero (<0.01): 68.3%
  % near one  (>0.99): 24.4%

H=72h:
  mean=0.3144  std=0.4641
  min=0.0000  max=1.0000
  % near zero (<0.01): 68.3%
  % near one  (>0.99): 31.2%

=== EVENT RATE BY HORIZON ===
H=12h: 49 hits / 215 known = 0.2279 event rate
H=24h: 63 hits / 196 known = 0.3214 event rate
H=48h: 66 hits / 166 known = 0.3976 event rate
H=72h: 69 hits / 221 known = 0.3122 event rate


In [7]:
# import sys
# print(sys.executable)
# print(sys.version)

# !{sys.executable} -m pip install -U pip xgboost

# ─────────────────────────────────────────────
# 5. XGBoost AFT SURVIVAL MODEL 
# ─────────────────────────────────────────────
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("xgboost not installed — pip install xgboost")
    HAS_XGB = False

if HAS_XGB:
    print("\n===== XGBoost AFT Survival Model =====")

    oof_surv  = pd.DataFrame(0.0, index=train_fe.index, columns=[f"prob_{H}h" for H in HORIZONS])
    test_surv = pd.DataFrame(0.0, index=test_fe.index,  columns=[f"prob_{H}h" for H in HORIZONS])

    XGB_PARAMS = {
        "objective": "survival:aft",
        "aft_loss_distribution": "normal",   # more stable than logistic on small data
        "aft_loss_distribution_scale": 1.0,
        "max_depth": 3,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "eval_metric": "aft-nloglik",
        "verbosity": 0,
        "seed": 42,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for H in HORIZONS:
        col = f"prob_{H}h"

        # ── Correct AFT labels ────────────────────────────────────
        # Hit cases:      lower = upper = observed time  (exact event)
        # Censored cases: lower = observed time, upper = +inf  (right-censored)
        # Both clipped to H since we only care about hits within H
        hit_mask = train_fe["event"] == 1

        y_lower = train_fe["time_to_hit_hours"].copy()
        y_upper = train_fe["time_to_hit_hours"].copy()

        # Censored: we know they didn't hit yet — lower=last observed, upper=inf
        y_upper[~hit_mask] = np.inf

        # Clip both to H — events after H become right-censored at H
        late_hit = hit_mask & (train_fe["time_to_hit_hours"] > H)
        y_lower[late_hit] = H
        y_upper[late_hit] = np.inf   # hit after H = censored for this horizon

        oof_preds  = np.zeros(len(train_fe))
        test_preds = np.zeros(len(test_fe))

        for fold, (tr_idx, va_idx) in enumerate(cv.split(train_fe, train_fe["event"])):
            dtrain = xgb.DMatrix(
                train_fe.iloc[tr_idx][FEATURE_COLS].values,
                label_lower_bound=y_lower.iloc[tr_idx].values,
                label_upper_bound=y_upper.iloc[tr_idx].values)
            dval = xgb.DMatrix(
                train_fe.iloc[va_idx][FEATURE_COLS].values,
                label_lower_bound=y_lower.iloc[va_idx].values,
                label_upper_bound=y_upper.iloc[va_idx].values)

            mdl = xgb.train(
                XGB_PARAMS, dtrain,
                num_boost_round=500,
                evals=[(dval, "val")],
                early_stopping_rounds=50,
                verbose_eval=False)

            # AFT predicts log(time) — convert to P(hit by H)
            # Lower predicted time = more likely to hit early
            log_pred_time = mdl.predict(dval)           # log scale
            pred_time     = np.exp(log_pred_time)       # back to hours
            # Probability: how much of the AFT distribution falls below H
            from scipy.stats import norm
            scale = XGB_PARAMS["aft_loss_distribution_scale"]
            oof_preds[va_idx] = norm.cdf(
                (np.log(H) - log_pred_time) / scale)

            log_test = mdl.predict(xgb.DMatrix(test_fe[FEATURE_COLS].values))
            test_preds += log_test

        # Average log predictions across folds then convert
        avg_log = test_preds / cv.get_n_splits()
        from scipy.stats import norm
        scale = XGB_PARAMS["aft_loss_distribution_scale"]
        test_surv[col] = norm.cdf((np.log(H) - avg_log) / scale)
        oof_surv[col]  = oof_preds

    oof_surv  = enforce_monotonic(oof_surv.clip(0, 1))
    test_surv = enforce_monotonic(test_surv.clip(0, 1))

    surv_wb, surv_briers = weighted_brier(train_fe, oof_surv)
    print(f"XGB AFT OOF WB: {surv_wb:.6f}")
    print(f"  Per horizon: { {k: round(v,6) for k,v in surv_briers.items()} }")
    print(f"  vs LGBM:     {lgbm_wb:.6f}")


===== XGBoost AFT Survival Model =====
XGB AFT OOF WB: 0.149719
  Per horizon: {24: 0.152588, 48: 0.162358, 72: 0.129996}
  vs LGBM:     0.011377


In [8]:
# ─────────────────────────────────────────────
# 6. BLEND LGBM + SURVIVAL
# ─────────────────────────────────────────────
if HAS_XGB:
    print("\n===== Blend Comparison =====")

    # Try three blend ratios — pick best on OOF
    blends = {
        "70 LGBM / 30 Surv": (0.70, 0.30),
        "60 LGBM / 40 Surv": (0.60, 0.40),
        "50 LGBM / 50 Surv": (0.50, 0.50),
        "80 LGBM / 20 Surv": (0.80, 0.20),
    }

    best_blend_wb   = lgbm_wb
    best_blend_name = "LGBM only"
    submission_probs = lgbm_test_bagged

    for name, (w_lgbm, w_surv) in blends.items():
        b_oof  = enforce_monotonic(
            lgbm_oof_bagged * w_lgbm + oof_surv * w_surv)
        b_test = enforce_monotonic(
            lgbm_test_bagged * w_lgbm + test_surv * w_surv)
        wb, briers = weighted_brier(train_fe, b_oof)
        marker = " ← best" if wb < best_blend_wb else ""
        print(f"  {name}: WB={wb:.6f}{marker}")
        if wb < best_blend_wb:
            best_blend_wb   = wb
            best_blend_name = name
            submission_probs = b_test

    print(f"\nSelected: {best_blend_name}  (WB={best_blend_wb:.6f})")
    print(f"vs LGBM alone: {lgbm_wb:.6f}")
else:
    submission_probs = lgbm_test_bagged


===== Blend Comparison =====
  70 LGBM / 30 Surv: WB=0.024105
  60 LGBM / 40 Surv: WB=0.033828
  50 LGBM / 50 Surv: WB=0.046292
  80 LGBM / 20 Surv: WB=0.017122

Selected: LGBM only  (WB=0.011377)
vs LGBM alone: 0.011377


In [9]:
# ─────────────────────────────────────────────
# 7. SUBMISSION
# ─────────────────────────────────────────────
submission = pd.DataFrame({
    "event_id": test["event_id"].values,
    "prob_12h": submission_probs["prob_12h"].values,
    "prob_24h": submission_probs["prob_24h"].values,
    "prob_48h": submission_probs["prob_48h"].values,
    "prob_72h": submission_probs["prob_72h"].values,
})
submission.to_csv("submission_v4_ensemble.csv", index=False)
print("\nSaved: submission_v4_ensemble.csv")
print(submission.describe())


Saved: submission_v4_ensemble.csv
           event_id   prob_12h   prob_24h   prob_48h   prob_72h
count  9.500000e+01  95.000000  95.000000  95.000000  95.000000
mean   5.695393e+07   0.180822   0.274606   0.290868   0.294737
std    2.329721e+07   0.304242   0.428033   0.452826   0.458343
min    1.066260e+07   0.000000   0.000000   0.000000   0.000000
25%    4.027536e+07   0.000000   0.000000   0.000000   0.000000
50%    5.480111e+07   0.000000   0.000000   0.000000   0.000000
75%    7.536942e+07   0.429119   0.846021   0.998429   1.000000
max    9.964946e+07   0.998039   1.000000   1.000000   1.000000
